# Lesson 07b Task: Neural Network Architecture Exploration
## Predicting Parkinson's Disease Severity from Voice

**Objective:** Try several different neural network architectures on a real medical dataset and see how the choices you make affect performance.

### The story behind the data

Parkinson's disease affects movement, but it also subtly affects the **voice** — patients tend to have slightly less steady pitch and volume when they hold a long sound like "aaaahhhh". Researchers ran a 6-month study with 42 Parkinson's patients who recorded themselves at home each week. From every recording they extracted 16 acoustic measurements (pitch wobble, volume wobble, breathiness, etc.), and they paired those measurements with the patient's UPDRS score — the standard 0–176 clinical score a neurologist gives to rate symptom severity.

**Your job:** Train a neural network that, given the voice measurements (plus age, sex, and time in the study), predicts the patient's UPDRS score.

> Why this matters: if a model can do this well, patients could check in from home every week instead of driving to a clinic every three months.

### Why neural networks for this?

The connection between voice acoustics and brain disease is not a straight line. Tiny pitch wobble might mean nothing on its own, but pitch wobble *plus* breathiness *plus* volume instability might mean a lot. Linear regression can't capture that kind of conditional, interacting pattern — but a neural network with hidden layers can.

**Data:** Parkinsons Telemonitoring (UCI id=189) — 5,875 voice recordings, 19 input features, target = `total_UPDRS`.

In [17]:
# Setup: import libraries and load the Parkinsons Telemonitoring dataset.
# We try the UCI repo first (recommended), then fall back to a local CSV
# if there's no internet.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# Load the data
try:
    from ucimlrepo import fetch_ucirepo
    parkinsons = fetch_ucirepo(id=189)
    parkinsonsData = parkinsons.data.features.copy()
    # The target columns live separately - merge them in
    parkinsonsData['motor_UPDRS'] = parkinsons.data.targets['motor_UPDRS']
    parkinsonsData['total_UPDRS'] = parkinsons.data.targets['total_UPDRS']
    print("Loaded from UCI repository.")
except Exception as e:
    print(f"UCI fetch failed ({e}). Falling back to local parkinsons_updrs.csv")
    parkinsonsData = pd.read_csv('parkinsons_updrs.csv')

print(f"\nData shape: {parkinsonsData.shape}")
print(f"\nColumns: {list(parkinsonsData.columns)}")
print(f"\nFirst few rows:")
print(parkinsonsData.head())


Loaded from UCI repository.

Data shape: (5875, 21)

Columns: ['age', 'test_time', 'Jitter(%)', 'Jitter(Abs)', 'Jitter:RAP', 'Jitter:PPQ5', 'Jitter:DDP', 'Shimmer', 'Shimmer(dB)', 'Shimmer:APQ3', 'Shimmer:APQ5', 'Shimmer:APQ11', 'Shimmer:DDA', 'NHR', 'HNR', 'RPDE', 'DFA', 'PPE', 'sex', 'motor_UPDRS', 'total_UPDRS']

First few rows:
   age  test_time  Jitter(%)  Jitter(Abs)  Jitter:RAP  Jitter:PPQ5  \
0   72     5.6431    0.00662     0.000034     0.00401      0.00317   
1   72    12.6660    0.00300     0.000017     0.00132      0.00150   
2   72    19.6810    0.00481     0.000025     0.00205      0.00208   
3   72    25.6470    0.00528     0.000027     0.00191      0.00264   
4   72    33.6420    0.00335     0.000020     0.00093      0.00130   

   Jitter:DDP  Shimmer  Shimmer(dB)  Shimmer:APQ3  ...  Shimmer:APQ11  \
0     0.01204  0.02565        0.230       0.01438  ...        0.01662   
1     0.00395  0.02024        0.179       0.00994  ...        0.01689   
2     0.00616  0.01675    

In [18]:
# Prepare features (X) and target (y)
#   Target: total_UPDRS (the doctor's 0-176 symptom severity score)
#   Features: everything except the two UPDRS scores
#     (age, sex, test_time, plus 16 voice-acoustic measurements = 19 features total)

X = parkinsonsData.drop(columns=['motor_UPDRS', 'total_UPDRS'])
y = parkinsonsData['total_UPDRS']

print(f"Features X: {X.shape[1]} columns, {X.shape[0]} rows")
print(f"Target y:   total_UPDRS, range {y.min():.1f} to {y.max():.1f}")

# Split into train (80%) and test (20%) - test stays locked away
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale features (REQUIRED for neural networks!)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nTraining set: {X_train.shape[0]} examples")
print(f"Test set:     {X_test.shape[0]} examples")


Features X: 19 columns, 5875 rows
Target y:   total_UPDRS, range 7.0 to 55.0

Training set: 4700 examples
Test set:     1175 examples


## A quick reminder before you start: early stopping is on

Every architecture below uses three parameters from the walkthrough:

```python
early_stopping=True       # hold out a validation chunk and watch it
validation_fraction=0.15  # 15% of training data goes to validation
n_iter_no_change=20       # stop if validation hasn't improved in 20 iterations
```

After fitting, look at:

- `model.n_iter_` — how many iterations actually ran (often way less than `max_iter`)
- `model.best_validation_score_` — the best R² the model ever hit on validation
- `model.loss_curve_` and `model.validation_scores_` — for plotting

When you compare architectures, **look at `n_iter_` too**. A bigger architecture that stops earlier is often a sign of overfitting — early stopping caught it in the act.

## Architecture 1 — Shallow Network - A Gimme

**Architecture:** 5 neurons (single hidden layer)

**Why test this?** A small network. Is 5 neurons enough capacity for this problem?

In [19]:
# Solution
model1 = MLPRegressor(
    hidden_layer_sizes=(5,),
    max_iter=2000,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=20,
    random_state=42,
)
model1.fit(X_train_scaled, y_train)

predictions1 = model1.predict(X_test_scaled)
r2_1 = r2_score(y_test, predictions1)
mae_1 = mean_absolute_error(y_test, predictions1)

print(f"Architecture 1:")
print(f"  R-squared:                 {r2_1:.4f}")
print(f"  MAE:                       {mae_1:.2f} UPDRS points")
print(f"  Stopped at iteration:      {model1.n_iter_} of {model1.max_iter}")
print(f"  Best validation R-squared: {model1.best_validation_score_:.4f}")



Architecture 1:
  R-squared:                 0.2226
  MAE:                       7.66 UPDRS points
  Stopped at iteration:      794 of 2000
  Best validation R-squared: 0.2158


## Architecture 2 — Medium Depth

**Architecture:** 32-16 neurons (two hidden layers)

**Why test this?** A moderate two-layer network. Should be a sensible default for this dataset.

In [24]:
# Build, train, and evaluate this architecture.
model2 = MLPRegressor(
    hidden_layer_sizes=(32, 16),
    max_iter=2000,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=20,
    random_state=42,
)
model2.fit(X_train_scaled, y_train)

predictions2 = model2.predict(X_test_scaled)
r2_2 = r2_score(y_test, predictions2)
mae_2 = mean_absolute_error(y_test, predictions2)

print(f"Architecture 2:")
print(f"  R-squared: {r2_2:.4f}")
print(f"  MAE: {mae_2:.2f} UPDRS points")
print(f"  Stopped at iteration:      {model2.n_iter_}")
print(f"  Best validation R-squared: {model2.best_validation_score_:.4f}")

Architecture 2:
  R-squared: 0.6585
  MAE: 4.38 UPDRS points
  Stopped at iteration:      853
  Best validation R-squared: 0.6777


## Architecture 3 — Wide and Shallow

**Architecture:** 64 neurons (single hidden layer)

**Why test this?** Lots of width but no depth. Tests width vs depth.

In [26]:
# Build, train, and evaluate this architecture.
model3 = MLPRegressor(
    hidden_layer_sizes=(64,),
    max_iter=2000,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=20,
    random_state=42,
)
model3.fit(X_train_scaled, y_train)

predictions3 = model3.predict(X_test_scaled)
r2_3 = r2_score(y_test, predictions3)
mae_3 = mean_absolute_error(y_test, predictions3)

print(f"Architecture 3:")
print(f"  R-squared: {r2_3:.4f}")
print(f"  MAE: {mae_3:.2f} UPDRS points")
print(f"  Stopped at iteration:      {model3.n_iter_}")
print(f"  Best validation R-squared: {model3.best_validation_score_:.4f}")


Architecture 3:
  R-squared: 0.6363
  MAE: 4.59 UPDRS points
  Stopped at iteration:      1123
  Best validation R-squared: 0.6519


## Architecture 4 — Deep Network

**Architecture:** 64-32-16 neurons (three hidden layers)

**Why test this?** Three layers, plenty of capacity. Watch n_iter_ — early stopping may fire fast.

In [27]:
# Build, train, and evaluate this architecture.
model4 = MLPRegressor(
    hidden_layer_sizes=(64, 32, 16),
    max_iter=2000,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=20,
    random_state=42,
)
model4.fit(X_train_scaled, y_train)

predictions4 = model4.predict(X_test_scaled)
r2_4 = r2_score(y_test, predictions4)
mae_4 = mean_absolute_error(y_test, predictions4)

print(f"Architecture 4:")
print(f"  R-squared: {r2_4:.4f}")
print(f"  MAE: {mae_4:.2f} UPDRS points")
print(f"  Stopped at iteration:      {model4.n_iter_}")
print(f"  Best validation R-squared: {model4.best_validation_score_:.4f}")


Architecture 4:
  R-squared: 0.7433
  MAE: 3.66 UPDRS points
  Stopped at iteration:      483
  Best validation R-squared: 0.7535


## Compare All Five

In [29]:
# Side-by-side comparison
print("\n" + "="*72)
print("ARCHITECTURE COMPARISON ON PARKINSONS TELEMONITORING")
print("="*72)
print(f"{'Architecture':<35} {'R-squared':<12} {'MAE':<10}")
print("-" * 72)
print(f"{'1. Shallow (5)':<35} {r2_1:<12.4f} {mae_1:<10.2f}")
print(f"{'2. Medium (32-16)':<35} {r2_2:<12.4f} {mae_2:<10.2f}")
print(f"{'3. Wide (64)':<35} {r2_3:<12.4f} {mae_3:<10.2f}")
print(f"{'4. Deep (64-32-16)':<35} {r2_4:<12.4f} {mae_4:<10.2f}")
print("="*72)

scores = [
    ('Architecture 1 (5)',        r2_1),
    ('Architecture 2 (32-16)',    r2_2),
    ('Architecture 3 (64)',       r2_3),
    ('Architecture 4 (64-32-16)', r2_4),
]
best_name, best_score = max(scores, key=lambda x: x[1])
print(f"\nBest performer: {best_name} with R-squared = {best_score:.4f}")



ARCHITECTURE COMPARISON ON PARKINSONS TELEMONITORING
Architecture                        R-squared    MAE       
------------------------------------------------------------------------
1. Shallow (5)                      0.2226       7.66      
2. Medium (32-16)                   0.6585       4.38      
3. Wide (64)                        0.6363       4.59      
4. Deep (64-32-16)                  0.7433       3.66      

Best performer: Architecture 4 (64-32-16) with R-squared = 0.7433


## Analysis Questions

1. **Which architecture performed best on the test data?** Did it match what you predicted? 
4 did the best. It had the highest r2 score and the lowest error. I kind of guessed it would win because it’s the most complex one, and it definitely handled the data better than the small ones.
2. **Which architecture stopped earliest (lowest `n_iter_`)?** Why might that be? What does it tell you about that architecture's relationship with this dataset?
Arch 4 actually stopped the earliest. It sounds weird because it’s the biggest model, but it basically got it faster than the others. It found the patterns quickly and stopped before it started making stuff up, which shows it’s really efficient for this data.
3. **Compare the wide network (Arch 3, single layer of 64) to the deep network (Arch 4, three layers of 64-32-16).** Same total neurons, very different shape. Which won? What does that suggest about depth vs width on this problem?
The deep one Arch 4 totally beat the wide one Arch 3. Even though they have a lot of neurons, stacking them in layers works way better than just having one giant row. It shows that for Parkinson’s scores, the model needs to process info in steps to actually be accurate.
4. **Look at the gap between `best_validation_score_` and your test R².** Are they similar? Bigger? Smaller? What does that tell you about how well early stopping picked the right moment to halt?
The validation and test scores were pretty much the same. This is good because it means the model wasn't just memorizing the practice test—it actually learned how to solve the problem. Early stopping basically pulled the plug at the perfect time.
5. **If you were going to deploy this model in a real Parkinson's monitoring app, which architecture would you choose, and why?** Consider not just R² but also training time and how well the model generalized.I’d definitely use Arch 4. If people are using an app to track their health, you want it to be as accurate as possible. It doesn't take that long to train anyway, so you might as well use the "smartest" version to get the best results.